In [0]:
#load data
import pandas as pd

df = spark.table("workspace.default.medicaid_enrollment").toPandas()

print(df.shape)
print(df.columns.tolist())
df.head()

(10302, 44)
['State Abbreviation', 'State Name', 'Reporting Period', 'State Expanded Medicaid', 'Preliminary or Updated', 'Final Report', 'New Applications Submitted to Medicaid and CHIP Agencies', 'New Applications Submitted to Medicaid and CHIP Agencies - footnotes', 'Applications for Financial Assistance Submitted to the State Based Marketplace', 'Applications for Financial Assistance Submitted to the State Based Marketplace - footnotes', 'Total Applications for Financial Assistance Submitted at State Level', 'Total Applications for Financial Assistance Submitted at State Level - footnotes', 'Individuals Determined Eligible for Medicaid at Application', 'Individuals Determined Eligible for Medicaid at Application - footnotes', 'Individuals Determined Eligible for CHIP at Application', 'Individuals Determined Eligible for CHIP at Application - footnotes', 'Total Medicaid and CHIP Determinations', 'Total Medicaid and CHIP Determinations - footnotes', 'Medicaid and CHIP Child Enrollmen

,State Abbreviation,State Name,Reporting Period,State Expanded Medicaid,Preliminary or Updated,Final Report,New Applications Submitted to Medicaid and CHIP Agencies,New Applications Submitted to Medicaid and CHIP Agencies - footnotes,Applications for Financial Assistance Submitted to the State Based Marketplace,Applications for Financial Assistance Submitted to the State Based Marketplace - footnotes,Total Applications for Financial Assistance Submitted at State Level,Total Applications for Financial Assistance Submitted at State Level - footnotes,Individuals Determined Eligible for Medicaid at Application,Individuals Determined Eligible for Medicaid at Application - footnotes,Individuals Determined Eligible for CHIP at Application,Individuals Determined Eligible for CHIP at Application - footnotes,Total Medicaid and CHIP Determinations,Total Medicaid and CHIP Determinations - footnotes,Medicaid and CHIP Child Enrollment,Medicaid and CHIP Child Enrollment - footnotes,Total Medicaid and CHIP Enrollment,Total Medicaid and CHIP Enrollment - footnotes,Total Medicaid Enrollment,Total Medicaid Enrollment - footnotes,Total CHIP Enrollment,Total CHIP Enrollment - footnotes,Total Adult Medicaid Enrollment,Total Adult Medicaid Enrollment - footnotes,Total Medicaid and CHIP Determinations Processed in Less than 24 Hours,Total Medicaid and CHIP Determinations Processed in Less than 24 Hours - footnotes,Total Medicaid and CHIP Determinations Processed Between 24 Hours and 7 Days,Total Medicaid and CHIP Determinations Processed Between 24 Hours and 7 Days - footnotes,Total Medicaid and CHIP Determinations Processed Between 8 Days and 30 Days,Total Medicaid and CHIP Determinations Processed Between 8 Days and 30 Days - footnotes,Total Medicaid and CHIP Determinations Processed between 31 days and 45 days,Total Medicaid and CHIP Determinations Processed between 31 days and 45 days - footnotes,Total Medicaid and CHIP Determinations Processed in More than 45 Days,Total Medicaid and CHIP Determinations Processed in More than 45 Days - footnotes,Total Call Center Volume (Number of Calls),Total Call Center Volume (Number of Calls) - footnotes,Average Call Center Wait Time (Minutes),Average Call Center Wait Time (Minutes) - footnotes,Average Call Center Abandonment Rate,Average Call Center Abandonment Rate - footnotes
0,AK,Alaska,201309,N,U,Y,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,122334.0,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
1,AK,Alaska,201706,Y,P,N,3017.0,Includes Renewals and/or Redeterminations,0.0,None,3017.0,Includes Renewals and/or Redeterminations,3422.0,Includes Renewals and/or Redeterminations; Inc...,0.0,None,3422.0,Includes Renewals and/or Redeterminations,88766.0,None,191171.0,None,179001.0,None,12170.0,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
2,AK,Alaska,201706,Y,U,Y,3069.0,Includes Renewals and/or Redeterminations,0.0,None,3069.0,Includes Renewals and/or Redeterminations,3422.0,Includes Renewals and/or Redeterminations; Inc...,0.0,None,3422.0,Includes Renewals and/or Redeterminations,90081.0,None,194534.0,None,182080.0,None,12454.0,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
3,AK,Alaska,201707,Y,P,N,3187.0,Includes Renewals and/or Redeterminations,0.0,None,3187.0,Includes Renewals and/or Redeterminations,3159.0,Includes Renewals and/or Redeterminations; Inc...,0.0,None,3159.0,Includes Renewals and/or Redeterminations,89358.0,None,193019.0,None,180641.0,None,12378.0,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
4,AK,Alaska,201707,Y,U,Y,3265.0,Includes Renewals and/or Redeterminations,0.0,None,3265.0,Includes Renewals and/or Redeterminations,3159.0,Includes Renewals and/or Redeterminations; Inc...,0.0,None,3159.0,Includes Renewals and/or Redeterminations,90562.0,None,196121.0,None,183516.0,None,12605.0,None,NaN,None,NaN,N

In [0]:
# clean
import numpy as np

cols = {
    'State Abbreviation': 'state',
    'Reporting Period': 'period',
    'Preliminary or Updated': 'p_or_u',
    'Total Medicaid and CHIP Enrollment': 'total_enrollment',
    'Total Medicaid and CHIP Enrollment - footnotes': 'total_footnote',
    'Total Medicaid Enrollment': 'medicaid_enrollment',
    'Total CHIP Enrollment': 'chip_enrollment',
}
clean = df[list(cols)].rename(columns=cols)

for c in ['total_enrollment', 'medicaid_enrollment', 'chip_enrollment']:
    clean[c] = pd.to_numeric(clean[c], errors='coerce')

clean['period'] = clean['period'].astype(str)
latest = clean['period'].max()
keep = (clean['p_or_u'] == 'U') | ((clean['p_or_u'] == 'P') & (clean['period'] == latest))
clean = clean[keep].copy()

clean['total_enrollment'] = clean['total_enrollment'].replace(0, np.nan)
clean['medicaid_enrollment'] = clean['medicaid_enrollment'].replace(0, np.nan)

clean['month'] = pd.to_datetime(clean['period'], format='%Y%m')
clean = clean.sort_values(['state', 'month']).reset_index(drop=True)

clean.head()

,state,period,p_or_u,total_enrollment,total_footnote,medicaid_enrollment,chip_enrollment,month
0,AK,201309,U,122334.0,None,NaN,NaN,2013-09-01
1,AK,201706,U,194534.0,None,182080.0,12454.0,2017-06-01
2,AK,201707,U,196121.0,None,183516.0,12605.0,2017-07-01
3,AK,201708,U,197834.0,None,185071.0,12763.0,2017-08-01
4,AK,201709,U,198742.0,None,185974.0,12768.0,2017-09-01


In [0]:
# check that cleaning worked
print("Rows:", clean.shape[0])
print(clean['p_or_u'].value_counts())
print("Duplicate state-months:", clean.duplicated(['state', 'period']).sum())
print("Missing total enrollment:", clean['total_enrollment'].isna().sum())
print(clean[clean['total_enrollment'].isna()][['state', 'period', 'total_footnote']])

Rows: 5202
p_or_u
U    5151
P      51
Name: count, dtype: int64
Duplicate state-months: 0
Missing total enrollment: 3
     state  period                                    total_footnote
612     CT  201309                                              None
2142    ME  201309                                              None
4069    RI  202412  Unable to Provide Data due to System Limitations


In [0]:
# save clean table
spark.createDataFrame(clean).write.mode("overwrite").saveAsTable("workspace.default.medicaid_enrollment_clean")

spark.table("workspace.default.medicaid_enrollment_clean").count()

5202

In [0]:
# national trend
trend = clean[clean['period'] >= '201706'].copy()

national = trend.groupby('month').agg(
    total_enrollment=('total_enrollment', 'sum'),
    states_reporting=('total_enrollment', 'count'),
).reset_index()

display(national)

month,total_enrollment,states_reporting
2017-06-01T00:00:00.000Z,7.3578497E7,51
2017-07-01T00:00:00.000Z,7.3538243E7,51
2017-08-01T00:00:00.000Z,7.3540423E7,51
2017-09-01T00:00:00.000Z,7.33766E7,51
2017-10-01T00:00:00.000Z,7.3523773E7,51
2017-11-01T00:00:00.000Z,7.3556074E7,51
2017-12-01T00:00:00.000Z,7.3669903E7,51
2018-01-01T00:00:00.000Z,7.3408315E7,51
2018-02-01T00:00:00.000Z,7.3517782E7,51
2018-03-01T00:00:00.000Z,7.3485554E7,51


In [0]:
# unusual month-to-month changes
trend['mom_pct'] = trend.groupby('state')['total_enrollment'].pct_change(fill_method=None) * 100

jumps = trend[trend['mom_pct'].abs() > 10][['state', 'period', 'total_enrollment', 'mom_pct']]
jumps.round(1)

,state,period,total_enrollment,mom_pct
1358,ID,202001,313122.0,17.0
1399,ID,202306,385133.0,-11.0
2833,NC,202312,2655404.0,13.5
3131,NH,202304,226579.0,-10.5
3722,OK,202107,1020015.0,13.7
4610,VA,201901,1237262.0,17.5
5175,WY,202308,76785.0,-10.5


In [0]:
# COVID growth vs. unwinding by state
key = trend[trend['period'].isin(['202002', '202304', '202510'])]
wide = key.pivot(index='state', columns='period', values='total_enrollment')

wide['covid_growth_pct'] = (wide['202304'] / wide['202002'] - 1) * 100
wide['unwinding_pct'] = (wide['202510'] / wide['202304'] - 1) * 100

wide.round(1).sort_values('unwinding_pct')

period,202002,202304,202510,covid_growth_pct,unwinding_pct
state,,,,,
MT,252740.0,330590.0,210942.0,30.8,-36.2
UT,313899.0,488964.0,332621.0,55.8,-32.0
CO,1266680.0,1713809.0,1191047.0,35.3,-30.5
TX,4198897.0,5950574.0,4148608.0,41.7,-30.3
WY,56082.0,85281.0,59714.0,52.1,-30.0
ID,319534.0,443371.0,312807.0,38.8,-29.4
FL,3600457.0,5118562.0,3648091.0,42.2,-28.7
GA,1833759.0,2542085.0,1872027.0,38.6,-26.4
LA,1495354.0,1921949.0,1421099.0,28.5,-26.1


In [0]:
# cleaning function
def clean_enrollment(data):
    data = data.copy()
    data['period'] = data['period'].astype(str)
    latest = data['period'].max()
    keep = (data['p_or_u'] == 'U') | ((data['p_or_u'] == 'P') & (data['period'] == latest))
    data = data[keep].copy()
    data['total_enrollment'] = data['total_enrollment'].replace(0, np.nan)
    return data

In [0]:
# test 
fake = pd.DataFrame({
    'state':            ['RI', 'RI', 'RI', 'RI', 'RI'],
    'period':           ['202411', '202411', '202412', '202412', '202501'],
    'p_or_u':           ['P', 'U', 'P', 'U', 'P'],
    'total_enrollment': [300, 305, 0, 0, 310],
})
result = clean_enrollment(fake)

def test_one_row_per_state_month():
    assert len(result) == 3
    assert result.duplicated(['state', 'period']).sum() == 0

def test_latest_month_keeps_preliminary():
    assert result[result['period'] == '202501']['p_or_u'].tolist() == ['P']

def test_zero_becomes_missing():
    assert result[result['period'] == '202412']['total_enrollment'].isna().all()

def test_real_data_complete():
    assert len(clean) == clean['state'].nunique() * clean['period'].nunique()

test_one_row_per_state_month()
test_latest_month_keeps_preliminary()
test_zero_becomes_missing()
test_real_data_complete()
print("All 4 tests passed ✅")

All 4 tests passed ✅
